# Amazon ML Challenge 2026: Business Entity Resolution (v7, experimental)

v7 = v6 + 'decoy business' features: how many Source 2/3 records share a record's name, name + house number,
and house number + street. On our development sample this added +0.001 F0.5 to stage 1 (0.9805 -> 0.9814).
Everything else as v6: 20 candidates, one-sided token features, per-source caps, v3-size model.

In [ ]:
import os
os.environ['ER_DATA_DIR'] = '/kaggle/input/datasets/urajkaushik/amazon-ml-challenge-private/dataset'
os.environ['ER_OUT_DIR'] = '/kaggle/working/output'
os.environ['ER_WORK_DIR'] = '/kaggle/working/work'
os.environ['ER_TMP_DIR'] = '/tmp'
os.makedirs(os.environ['ER_TMP_DIR'], exist_ok=True)
os.environ['ER_TOPK'] = '20'
print('DATA:', os.environ['ER_DATA_DIR'])
!free -g

In [ ]:
import os, re, zlib, time, gc, pickle, unicodedata, collections, tempfile
import numpy as np, pandas as pd, scipy.sparse as sp
from joblib import Parallel, delayed
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn.ensemble import HistGradientBoostingClassifier

# ---------------- configuration ----------------
DATA_DIR = os.environ.get('ER_DATA_DIR', 'dataset')        # folder containing train/ and test/
OUT_DIR = os.environ.get('ER_OUT_DIR', 'output')
WORK_DIR = os.environ.get('ER_WORK_DIR', 'work')           # models, threshold, translit dictionary
N_JOBS = int(os.environ.get('ER_N_JOBS', os.cpu_count() or 1))
TRAIN_S1_FRAC = float(os.environ.get('ER_TRAIN_S1_FRAC', 0.03))  # share of train S1 entities used for model rows + validation
MAX_ROWS_PER_FOLD = int(os.environ.get('ER_MAX_ROWS', 3_000_000))
MAX_DF_FRAC, MIN_MAX_DF = 2.7e-4, 50   # blocking keys shared by more S1 records than this are dropped
TOPK = int(os.environ.get('ER_TOPK', 20))   # v6: 20 candidates per record (v4's recall change)
REL_PRUNE = 0.30
TEST_MODELS = int(os.environ.get('ER_TEST_MODELS', 1))   # v4: 1 model per stage at test time (halves prediction time)
VARIANT_THRS = [0.80, 0.85, 0.90, 0.93, 0.95]   # extra matching files written for leaderboard threshold tests
THR_FRANCE = float(os.environ['ER_THR_FRANCE']) if os.environ.get('ER_THR_FRANCE') else None  # optional separate threshold for France (no labels to tune it)             # candidates per S2/S3 record, relative blocking-score pruning
GBM_PARAMS = dict(max_iter=300, learning_rate=0.1, max_leaf_nodes=63, min_samples_leaf=40,   # v3-size model (keeps run time close to v4)
                  l2_regularization=1.0, early_stopping=False, random_state=42)
TMP_DIR = os.environ.get('ER_TMP_DIR', tempfile.gettempdir())   # big feature matrices are memory-mapped here, not held in RAM
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(WORK_DIR, exist_ok=True)
T0 = time.time()


def log(*a):
    print(f'[{time.time() - T0:7.0f}s]', *a, flush=True)


def read_tsv(path):
    return pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)

log('N_JOBS =', N_JOBS)

## 1. Normalization (transliteration, names, addresses)

In [ ]:
import re, unicodedata
import numpy as np

# ---------------------------------------------------------------------------
# 1. Brahmic (Indic) script -> Latin transliteration.
#    All major Indic Unicode blocks share the ISCII layout, so one table keyed by
#    the offset inside the block handles Devanagari, Bengali, Gurmukhi, Gujarati,
#    Oriya, Tamil, Telugu, Kannada and Malayalam.
# ---------------------------------------------------------------------------
_BLOCKS = [0x0900, 0x0980, 0x0A00, 0x0A80, 0x0B00, 0x0B80, 0x0C00, 0x0C80, 0x0D00]
_VOW = {0x05: 'a', 0x06: 'a', 0x07: 'i', 0x08: 'i', 0x09: 'u', 0x0A: 'u', 0x0B: 'ri', 0x0C: 'li',
        0x0D: 'e', 0x0E: 'e', 0x0F: 'e', 0x10: 'ai', 0x11: 'o', 0x12: 'o', 0x13: 'o', 0x14: 'au'}
_CON = {0x15: 'k', 0x16: 'kh', 0x17: 'g', 0x18: 'gh', 0x19: 'n', 0x1A: 'ch', 0x1B: 'chh', 0x1C: 'j',
        0x1D: 'jh', 0x1E: 'n', 0x1F: 't', 0x20: 'th', 0x21: 'd', 0x22: 'dh', 0x23: 'n', 0x24: 't',
        0x25: 'th', 0x26: 'd', 0x27: 'dh', 0x28: 'n', 0x29: 'n', 0x2A: 'p', 0x2B: 'f', 0x2C: 'b',
        0x2D: 'bh', 0x2E: 'm', 0x2F: 'y', 0x30: 'r', 0x31: 'r', 0x32: 'l', 0x33: 'l', 0x34: 'l',
        0x35: 'v', 0x36: 'sh', 0x37: 'sh', 0x38: 's', 0x39: 'h',
        0x58: 'q', 0x59: 'kh', 0x5A: 'g', 0x5B: 'z', 0x5C: 'd', 0x5D: 'dh', 0x5E: 'f', 0x5F: 'y'}
_MAT = {0x3E: 'a', 0x3F: 'i', 0x40: 'i', 0x41: 'u', 0x42: 'u', 0x43: 'ri', 0x44: 'ri', 0x45: 'e',
        0x46: 'e', 0x47: 'e', 0x48: 'ai', 0x49: 'o', 0x4A: 'o', 0x4B: 'o', 0x4C: 'au', 0x55: '', 0x56: 'ai', 0x57: 'au'}
_NAS = {0x01: 'n', 0x02: 'n', 0x03: 'h'}
_VIRAMA, _NUKTA = 0x4D, 0x3C


def _indic_off(ch):
    o = ord(ch)
    if 0x0900 <= o < 0x0D80:
        b = _BLOCKS[(o - 0x0900) // 0x80]
        return o - b
    return None


def translit(s):
    """Transliterate any Brahmic script characters in s to rough Latin; leave the rest untouched."""
    if not s or all(ord(c) < 0x0900 or ord(c) >= 0x0D80 for c in s):
        return s
    out, pend = [], False  # pend = consonant waiting for its inherent vowel
    for ch in s:
        off = _indic_off(ch)
        if off is None:
            if pend:
                pend = False  # word-final schwa deletion
            if ch in '\u200c\u200d':
                continue
            out.append(ch)
            continue
        if off in _CON:
            if pend:
                out.append('a')
            out.append(_CON[off]); pend = True
        elif off in _MAT:
            out.append(_MAT[off]); pend = False
        elif off == _VIRAMA:
            pend = False
        elif off == _NUKTA:
            continue
        elif off in _VOW:
            if pend:
                out.append('a'); pend = False
            out.append(_VOW[off])
        elif off in _NAS:
            if pend:
                out.append('a'); pend = False
            out.append(_NAS[off])
        elif 0x66 <= off <= 0x6F:
            if pend:
                out.append('a'); pend = False
            out.append(str(off - 0x66))
        elif off == 0x64 or off == 0x65:
            pend = False; out.append(' ')
        # everything else (signs we do not model) is dropped
    return ''.join(out)


def ascii_fold(s):
    s = unicodedata.normalize('NFKD', s)
    return ''.join(c for c in s if not unicodedata.combining(c))


def base_clean(s):
    s = unicodedata.normalize('NFKC', s or '')
    s = translit(s)
    s = ascii_fold(s).lower()
    return s

# ---------------------------------------------------------------------------
# 2. Business names
# ---------------------------------------------------------------------------
LEGAL = set('''inc incorporated llc l.l.c llp lp pllc pc corp corporation co company ltd limited pvt private
plc gmbh sarl sas sasu eurl sa sci snc scop the and of a an & pvtltd privatelimited ltda pty'''.split())
# frequent transliteration outputs of Indian legal words (produced by translit())
LEGAL |= set('praivet praiveta limited limitd limitad pra li pvt'.split())
_web_re = re.compile(r'(?:https?://)?(?:www\.)?([a-z0-9\-]+)\.(?:co\.in|com|in|net|org|fr|co|biz|info|us)\b')
_dba_re = re.compile(r'^(.*?)\b(?:dba|aka|d/b/a|t/a)\b:?(.*)$')
_nonalnum = re.compile(r'[^a-z0-9 ]+')
_digit_word = str.maketrans({'0': 'o', '1': 'l', '3': 'e', '5': 's'})


def norm_name(raw, allow_dba=True):
    """Return (clean_name, core_tokens(list), website_core(str), had_dba(int))."""
    s = base_clean(raw)
    web = ''
    m = _web_re.search(s)
    if m:
        web = m.group(1).replace('-', '')
        s = _web_re.sub(' ', s)
    s = s.replace('|', ' ')
    dba = 0
    m = _dba_re.match(s)
    if allow_dba and m and m.group(2).strip():
        s = m.group(2); dba = 1
    s = s.replace('&', ' and ').replace("'", '').replace('.', '')
    s = _nonalnum.sub(' ', s)
    toks = []
    for t in s.split():
        # fix digit-for-letter typos inside mostly-alphabetic words (F0ods -> foods)
        if any(c.isdigit() for c in t) and sum(c.isalpha() for c in t) >= max(2, len(t) - 1):
            t = t.translate(_digit_word)
        toks.append(t)
    clean = ' '.join(toks)
    core = [t for t in toks if t not in LEGAL]
    if not core:
        core = toks[:]
    if not clean and web:
        clean = web
    return clean, core, web, dba

# ---------------------------------------------------------------------------
# 3. Addresses
# ---------------------------------------------------------------------------
US_STATES = {'alabama': 'al', 'alaska': 'ak', 'arizona': 'az', 'arkansas': 'ar', 'california': 'ca', 'colorado': 'co',
             'connecticut': 'ct', 'delaware': 'de', 'florida': 'fl', 'georgia': 'ga', 'hawaii': 'hi', 'idaho': 'id',
             'illinois': 'il', 'indiana': 'in', 'iowa': 'ia', 'kansas': 'ks', 'kentucky': 'ky', 'louisiana': 'la',
             'maine': 'me', 'maryland': 'md', 'massachusetts': 'ma', 'michigan': 'mi', 'minnesota': 'mn',
             'mississippi': 'ms', 'missouri': 'mo', 'montana': 'mt', 'nebraska': 'ne', 'nevada': 'nv',
             'new hampshire': 'nh', 'new jersey': 'nj', 'new mexico': 'nm', 'new york': 'ny', 'north carolina': 'nc',
             'north dakota': 'nd', 'ohio': 'oh', 'oklahoma': 'ok', 'oregon': 'or', 'pennsylvania': 'pa',
             'rhode island': 'ri', 'south carolina': 'sc', 'south dakota': 'sd', 'tennessee': 'tn', 'texas': 'tx',
             'utah': 'ut', 'vermont': 'vt', 'virginia': 'va', 'washington': 'wa', 'west virginia': 'wv',
             'wisconsin': 'wi', 'wyoming': 'wy', 'district of columbia': 'dc'}
IN_STATES = {'andhra pradesh': 'ap', 'arunachal pradesh': 'ar', 'assam': 'as', 'bihar': 'br', 'chhattisgarh': 'cg',
             'goa': 'ga', 'gujarat': 'gj', 'haryana': 'hr', 'himachal pradesh': 'hp', 'jharkhand': 'jh',
             'karnataka': 'ka', 'kerala': 'kl', 'madhya pradesh': 'mp', 'maharashtra': 'mh', 'manipur': 'mn',
             'meghalaya': 'ml', 'mizoram': 'mz', 'nagaland': 'nl', 'odisha': 'od', 'orissa': 'od', 'punjab': 'pb',
             'rajasthan': 'rj', 'sikkim': 'sk', 'tamil nadu': 'tn', 'telangana': 'tg', 'tripura': 'tr',
             'uttar pradesh': 'up', 'uttarakhand': 'uk', 'uttaranchal': 'uk', 'west bengal': 'wb', 'delhi': 'dl',
             'new delhi': 'dl', 'jammu and kashmir': 'jk', 'chandigarh': 'ch', 'puducherry': 'py', 'pondicherry': 'py',
             # transliterated native-script spellings seen in the data
             'telangaana': 'tg', 'telanganaa': 'tg', 'maharashtr': 'mh', 'maharashtra': 'mh', 'karnatak': 'ka',
             'karnaatak': 'ka', 'tamilnadu': 'tn', 'tamizh nadu': 'tn', 'bihaar': 'br', 'dilli': 'dl',
             'uttar pradesh': 'up', 'gujaraat': 'gj', 'keral': 'kl', 'pashchim bangal': 'wb', 'paschim banga': 'wb',
             'rajasthaan': 'rj', 'panjab': 'pb', 'telangan': 'tg', 'andhrapradesh': 'ap', 'pashchimabang': 'wb',
             'madhy pradesh': 'mp', 'hariyana': 'hr', 'pajab': 'pb', 'tamilnadu': 'tn', 'keralam': 'kl', 'odisha': 'od'}
IN_STATE_ALT = {'ts': 'tg', 'or': 'od', 'ct': 'cg', 'ut': 'uk', 'dl': 'dl'}
CITY_ALIAS = {'bengaluru': 'bangalore', 'bengaluru urban': 'bangalore', 'madras': 'chennai', 'bombay': 'mumbai',
              'calcutta': 'kolkata', 'gurugram': 'gurgaon', 'poona': 'pune', 'mysuru': 'mysore', 'baroda': 'vadodara',
              'cochin': 'kochi', 'trivandrum': 'thiruvananthapuram', 'benares': 'varanasi', 'banaras': 'varanasi',
              'prayagraj': 'allahabad', 'mangaluru': 'mangalore', 'belagavi': 'belgaum', 'kalyan': 'kalyan'}
STREET = {'street': 'st', 'str': 'st', 'saint': 'st', 'road': 'rd', 'avenue': 'ave', 'av': 'ave', 'drive': 'dr',
          'lane': 'ln', 'court': 'ct', 'circle': 'cir', 'boulevard': 'blvd', 'highway': 'hwy', 'parkway': 'pkwy',
          'trail': 'trl', 'tr': 'trl', 'terrace': 'ter', 'square': 'sq', 'place': 'pl', 'apartment': 'apt',
          'suite': 'ste', 'north': 'n', 'south': 's', 'east': 'e', 'west': 'w', 'mount': 'mt', 'fort': 'ft',
          'point': 'pt', 'expressway': 'expy', 'freeway': 'fwy', 'crossing': 'xing', 'heights': 'hts',
          'number': 'no', 'nagar': 'nagar', 'rue': 'rue', 'boulevard': 'blvd', 'bd': 'blvd', 'avenue': 'ave',
          'chemin': 'chem', 'allee': 'all', 'impasse': 'imp', 'route': 'rte', 'first': '1st', 'second': '2nd',
          'third': '3rd', 'fourth': '4th', 'fifth': '5th'}

# France (no training labels): street abbreviations and department -> region, hand-coded
FR_STREET = {'r': 'rue', 'bld': 'blvd', 'ch': 'chem', 'qu': 'quai', 'crs': 'cours', 'fg': 'fbg', 'faubourg': 'fbg',
             'sainte': 'st', 'ste': 'st', 'lotissement': 'lot', 'residence': 'res', 'domaine': 'dom', 'cedex': '', 'france': ''}
_FR_REG = {'hdf': 'hauts de france|nord|pas de calais|somme|oise|aisne',
           'naq': 'nouvelle aquitaine|gironde|landes|pyrenees atlantiques|lot et garonne|dordogne|charente maritime|charente|deux sevres|haute vienne|vienne|creuse|correze',
           'pdl': 'pays de la loire|loire atlantique|maine et loire|mayenne|sarthe|vendee',
           'idf': 'ile de france|seine saint denis|seine et marne|hauts de seine|val de marne|val d oise|yvelines|essonne',
           'bre': 'bretagne', 'nor': 'normandie', 'ges': 'grand est', 'bfc': 'bourgogne franche comte',
           'cvl': 'centre val de loire', 'ara': 'auvergne rhone alpes', 'occ': 'occitanie',
           'pac': 'provence alpes cote d azur', 'cor': 'corse'}
FR_REGION = {n: code for code, names in _FR_REG.items() for n in names.split('|')}
_fr_re = re.compile(r'\b(' + '|'.join(sorted(map(re.escape, FR_REGION), key=len, reverse=True)) + r')\b')
ADDR_DROP = set('null none na nil city township cdp town village unit near opp opposite behind beside po box dist district'.split())
_state_re = None


def _build_state_re():
    names = sorted(set(US_STATES) | set(IN_STATES) | set(CITY_ALIAS), key=len, reverse=True)
    return re.compile(r'\b(' + '|'.join(re.escape(n) for n in names) + r')\b')


_frac = re.compile(r'(?<![\d/-])[1-3]/[2-4](?![\d/-])')
_addr_non = re.compile(r'[^a-z0-9 ]+')
_num_re = re.compile(r'\d+')


def norm_addr(raw, country):
    """Return (clean_addr_tokens(list), numbers(list of str), digit_concat(str), state(str))."""
    global _state_re
    if _state_re is None:
        _state_re = _build_state_re()
    s = base_clean(raw)
    s = s.replace('<null>', ' ').replace('1/2', ' ')
    s = _frac.sub(' ', s)
    cty = (country or '').lower()
    state = ''

    def rep(m):
        nonlocal state
        w = m.group(1)
        if w in CITY_ALIAS:
            return ' ' + CITY_ALIAS[w] + ' '
        if cty == 'us' and w in US_STATES:
            state = US_STATES[w]; return ' ' + state + ' '
        if cty == 'india' and w in IN_STATES:
            state = IN_STATES[w]; return ' ' + state + ' '
        return w
    s = _state_re.sub(rep, s)
    if cty == 'france':
        s = s.replace('-', ' ').replace("'", ' ')
        def rep_fr(m):
            nonlocal state
            state = FR_REGION[m.group(1)]; return ' ' + state + ' '
        s = _fr_re.sub(rep_fr, s)
    # numbers: hyphen/slash groups as a whole and as parts
    nums = []
    for grp in re.findall(r'\d[\d\-/]*\d|\d', s):
        parts = [p.lstrip('0') or '0' for p in _num_re.findall(grp)]
        nums.extend(parts)
        if len(parts) > 1:
            nums.append(''.join(parts))
    digit_concat = ''.join(_num_re.findall(s)).lstrip('0')
    comps = []
    for comp in s.split(','):
        comp = _addr_non.sub(' ', comp)
        ct = []
        for t in comp.split():
            if t.isdigit():
                t = t.lstrip('0') or '0'
            elif t[0].isdigit():
                m = re.match(r'^(\d+)([a-z]+)$', t)
                if m and m.group(2) not in ('st', 'nd', 'rd', 'th'):
                    ct.append(m.group(1).lstrip('0') or '0'); t = m.group(2)
            t = STREET.get(t, t)
            if cty == 'india' and t in IN_STATE_ALT:
                t = IN_STATE_ALT[t]
            if cty == 'france':
                t = FR_STREET.get(t, t)
            if not t or t in ADDR_DROP:
                continue
            ct.append(t)
        if ct:
            comps.append(ct)
    toks = [t for c in comps for t in c]
    if not state and toks:
        cand = set(US_STATES.values()) if cty == 'us' else set(IN_STATES.values()) if cty == 'india' else set()
        for t in toks:
            if t in cand:
                state = t
    return toks, nums, digit_concat, state, comps

## 2. Record preparation and blocking keys
Each record gets ~17 hashed keys: name tokens, sorted name-token pairs, concatenated name / website prefixes,
(house number, street word) pairs, adjacent address word pairs, number groups.

In [ ]:
NATIVE = re.compile(r'[\u0900-\u0D7F]')


def rec_keys(core, web, comps, dcat, nums=()):
    K = set()
    for t in core:
        if len(t) >= 2: K.add('n' + t)
    u = list(dict.fromkeys(core))[:6]
    for i in range(len(u)):
        for j in range(i + 1, len(u)):
            a, b = sorted((u[i], u[j])); K.add('p' + a + ' ' + b)
    cc = ''.join(core)
    if len(cc) >= 4: K.add('c' + cc); K.add('x' + cc[:6])
    if len(web) >= 4: K.add('c' + web); K.add('x' + web[:6])
    for comp in comps:
        nums_c = [t for t in comp if t.isdigit()]; words = [t for t in comp if not t.isdigit() and len(t) > 1]
        for n in nums_c[:4]:
            for w in words[:6]: K.add('a' + n + ' ' + w)
        for i in range(len(words) - 1): K.add('b' + words[i] + ' ' + words[i + 1])
        for w in words:
            if len(w) >= 3: K.add('w' + w)
    if len(dcat) >= 3: K.add('d' + dcat)
    for n in nums:
        if len(n) >= 3: K.add('g' + n)
    return np.array([zlib.crc32(k.encode()) for k in K], dtype=np.uint32)


def _prep_chunk(df, src, tdict):
    out = {'id': df.entity_id.values, 'src': np.full(len(df), src, np.int8), 'country': df.country.str.lower().values}
    native = df.business_name.str.contains(NATIVE).values
    nm = [norm_name(n, allow_dba=(src != 1)) for n in df.business_name.values]
    if tdict:
        nm = [(' '.join(tdict.get(w, w) for w in x[0].split()), [tdict.get(w, w) for w in x[1]], x[2], x[3]) if nv else x
              for x, nv in zip(nm, native)]
    out['name'] = [x[0] for x in nm]; out['core'] = [' '.join(x[1]) for x in nm]
    out['web'] = [x[2] for x in nm]; out['dba'] = np.array([x[3] for x in nm], np.int8)
    out['native'] = native.astype(np.int8)
    ad = [norm_addr(a, c) for a, c in zip(df.business_address.values, df.country.values)]
    out['addr'] = [' '.join(x[0]) for x in ad]; out['nums'] = [' '.join(x[1]) for x in ad]
    out['dcat'] = [x[2] for x in ad]; out['state'] = [x[3] for x in ad]
    ks = [rec_keys(n[1], n[2], a[4], a[2], a[1]) for n, a in zip(nm, ad)]
    out['nkeys'] = np.array([len(k) for k in ks], np.int32)
    return pd.DataFrame(out), (np.concatenate(ks) if ks else np.zeros(0, np.uint32))


def prep_all(raw, tdict, chunk=100_000):
    """raw: {1: df, 2: df, 3: df}. Returns R (S1 rows first), flat keys, n1."""
    Rs, Ks = [], []
    for src in (1, 2, 3):
        df = raw[src]
        parts = Parallel(n_jobs=N_JOBS)(delayed(_prep_chunk)(df.iloc[i:i + chunk], src, tdict)
                                        for i in range(0, len(df), chunk))
        Rs += [p[0] for p in parts]; Ks += [p[1] for p in parts]
        log(f'  prepared source {src}: {len(df):,} records')
    R = pd.concat(Rs, ignore_index=True)
    return R, np.concatenate(Ks), int((R.src == 1).sum())


def learn_tdict(S1raw, Qraw, gt, min_count=2, min_share=0.6):
    """Token dictionary transliterated-word -> English word, learned from native-script S2/S3 names
    in the training ground truth (e.g. 'praivet'->'private', 'kansaltents'->'consultants')."""
    nat = Qraw[Qraw.business_name.str.contains(NATIVE)]
    natname = dict(zip(nat.entity_id, nat.business_name))
    s1name = dict(zip(S1raw.entity_id, S1raw.business_name))
    cnt = collections.defaultdict(collections.Counter)
    for s, ms in zip(gt.source1_entity_id.values, gt.matched_entity_ids.values):
        a = None
        for m in ms.split(','):
            if m in natname:
                if a is None: a = norm_name(s1name[s], False)[0].split()
                b = norm_name(natname[m])[0].split()
                if len(a) == len(b):
                    for x, y in zip(b, a):
                        if x != y: cnt[x][y] += 1
    D = {}
    for x, c in cnt.items():
        y, n = c.most_common(1)[0]
        if n >= min_count and n >= min_share * sum(c.values()): D[x] = y
    return D

## 3. Blocking: sparse key-overlap top-k (each S2/S3 record retrieves its best S1 candidates)

In [ ]:
def build_index(keys1, nk1, max_df):
    uk, inv, df = np.unique(keys1, return_inverse=True, return_counts=True)
    ok = df <= max_df
    N1 = len(nk1)
    idf = (np.log1p(N1 / df) * ok).astype(np.float32)
    rows = np.repeat(np.arange(N1), nk1)
    m = ok[inv]
    A = sp.csr_matrix((np.ones(int(m.sum()), np.float32), (inv[m], rows[m])), shape=(len(uk), N1))
    W1 = np.asarray(A.T @ idf).ravel()
    return uk, idf, A, np.sqrt(W1 + 1.0).astype(np.float32)


def topk_rows(S, k):
    n = S.shape[0]; cnt = np.diff(S.indptr)
    rows = np.repeat(np.arange(n), cnt)
    order = np.lexsort((-S.data, rows))
    r = rows[order]; c = S.indices[order]; v = S.data[order]
    rank = np.arange(len(r)) - S.indptr[r]
    keep = rank < k
    return r[keep], c[keep], v[keep], rank[keep]


def _block_chunk(uk, idf, A, W1a, kk, nk, k, rel_prune):
    pos = np.searchsorted(uk, kk); pos[pos >= len(uk)] = 0
    hit = uk[pos] == kk
    rows = np.repeat(np.arange(len(nk)), nk)
    w = idf[pos] * hit; m = w > 0
    Q = sp.csr_matrix((w[m], (rows[m], pos[m])), shape=(len(nk), len(uk)))
    S = (Q @ A).tocsr()
    S.data = S.data / W1a[S.indices]          # length-normalised overlap score
    r, c, v, rk = topk_rows(S, k)
    first = np.r_[True, r[1:] != r[:-1]]
    grp = np.cumsum(first) - 1
    rel = v / v[np.flatnonzero(first)][grp]   # score relative to this record's best candidate
    keep = rel >= rel_prune
    return r[keep], c[keep], v[keep], rk[keep], rel[keep]


def run_blocking(R, keys, n1, chunk=20_000):
    nk = R.nkeys.values; offs = np.concatenate([[0], np.cumsum(nk)]).astype(np.int64)
    max_df = max(MIN_MAX_DF, int(MAX_DF_FRAC * n1))
    uk, idf, A, W1a = build_index(keys[:offs[n1]], nk[:n1], max_df)
    log(f'  index: {len(uk):,} keys, max_df={max_df}')
    starts = list(range(n1, len(R), chunk))
    res = Parallel(n_jobs=N_JOBS)(delayed(_block_chunk)(uk, idf, A, W1a, keys[offs[s]:offs[min(s + chunk, len(R))]],
                                                         nk[s:min(s + chunk, len(R))], TOPK, REL_PRUNE) for s in starts)
    q = np.concatenate([r[0] + s for r, s in zip(res, starts)]).astype(np.int32)
    s1 = np.concatenate([r[1] for r in res]).astype(np.int32)
    bn = np.concatenate([r[2] for r in res]).astype(np.float32)
    rk = np.concatenate([r[3] for r in res]).astype(np.int16)
    rl = np.concatenate([r[4] for r in res]).astype(np.float32)
    del res
    C = pd.DataFrame({'q': q, 's1': s1, 'bscore': bn * W1a[s1], 'bnorm': bn, 'brank': rk, 'rel': rl})
    log(f'  candidate pairs (top-{TOPK}, pruned): {len(C):,}')
    return C


def add_ctx(C):
    g = C.groupby('q')
    C['q_n'] = g.s1.transform('size').astype(np.int16)
    C['q_top'] = g.bnorm.transform('max')
    r = g.bnorm.rank(ascending=False, method='first')
    C['q_second'] = C.bnorm.where(r == 2, 0.0).groupby(C.q).transform('max')
    C['gap'] = np.where(C.brank == 0, C.bnorm - C.q_second, C.bnorm - C.q_top).astype(np.float32)
    C['s1_nq'] = C.groupby('s1').q.transform('size').astype(np.int32)
    C['s1_nq0'] = (C.brank == 0).astype(np.int32).groupby(C.s1).transform('sum')
    return C

CTX_COLS = ['bscore', 'bnorm', 'brank', 'rel', 'q_n', 'q_top', 'q_second', 'gap', 's1_nq', 's1_nq0']

## 4. Pairwise features (vectorised with hashed sparse vectors)

In [ ]:
HV_C3 = HashingVectorizer(analyzer='char_wb', ngram_range=(3, 3), n_features=2**20, alternate_sign=False, norm='l2', lowercase=False)
HV_C3n = HashingVectorizer(analyzer='char', ngram_range=(3, 3), n_features=2**20, alternate_sign=False, norm='l2', lowercase=False)
HV_TOK = HashingVectorizer(analyzer=str.split, n_features=2**22, alternate_sign=False, norm=None, binary=True, lowercase=False)
FEAT_COLS = ['name', 'core', 'web', 'addr', 'nums', 'dcat', 'state', 'native', 'dba', 'src', 'chain', 'fnum', 'cat',
             'qcnt_name', 'qcnt_sig', 'qcnt_addr']


def rowdot(A, B):
    return np.asarray(A.multiply(B).sum(1)).ravel().astype(np.float32)


def tok_idf(strings, n_docs):
    X = HV_TOK.transform(strings)
    df = np.bincount(X.indices, minlength=X.shape[1]).astype(np.float32)
    return np.log1p(n_docs / (df + 1.0)).astype(np.float32)


def tok_known(strings):
    """1.0 for hashed tokens that occur in at least one Source 1 record (a real vocabulary word), else 0."""
    X = HV_TOK.transform(strings)
    return (np.bincount(X.indices, minlength=X.shape[1]) > 0).astype(np.float32)


def only_feats(Xa, Xb, idf, known, prefix):
    """v5: tokens present on only one side. Typos create words never seen in Source 1 (oov);
    planted decoys swap in real words (known). Distinguishing the two is the main precision problem."""
    Bo = (Xb - Xb.multiply(Xa)).tocsr(); Bo.eliminate_zeros()
    Ao = (Xa - Xa.multiply(Xb)).tocsr(); Ao.eliminate_zeros()
    nb_only = np.asarray(Bo.sum(1)).ravel()
    kn = np.asarray(Bo @ known).ravel()
    return {prefix + 'b_oov': (nb_only - kn).astype(np.float32), prefix + 'b_known': kn.astype(np.float32),
            prefix + 'b_known_maxidf': Bo.multiply(idf * known).max(1).toarray().ravel().astype(np.float32),
            prefix + 'a_only_maxidf': Ao.multiply(idf).max(1).toarray().ravel().astype(np.float32)}


def set_feats(Xa, Xb, idf, prefix):
    inter = rowdot(Xa, Xb)
    na = np.asarray(Xa.sum(1)).ravel().astype(np.float32); nb = np.asarray(Xb.sum(1)).ravel().astype(np.float32)
    D = sp.diags(idf)
    wa = np.asarray((Xa @ D).sum(1)).ravel(); wb = np.asarray((Xb @ D).sum(1)).ravel()
    wi = np.asarray((Xa.multiply(Xb) @ D).sum(1)).ravel()
    return {prefix + 'inter': inter, prefix + 'na': na, prefix + 'nb': nb,
            prefix + 'jac': inter / np.maximum(na + nb - inter, 1),
            prefix + 'wja': wi / np.maximum(wa, 1e-6), prefix + 'wjb': wi / np.maximum(wb, 1e-6),
            prefix + 'wmiss_a': wa - wi, prefix + 'wmiss_b': wb - wi}


def lcp(a, b):
    n = min(len(a), len(b)); i = 0
    while i < n and a[i] == b[i]: i += 1
    return i


def first_num(s):
    for t in s.split():
        return int(t[:9])
    return -1


def add_stats(R, n1):
    """Global statistics used by the features (computed on Source 1 only)."""
    S1 = R.iloc[:n1]
    idfs = (tok_idf(S1.core.values, n1), tok_idf(S1.addr.values, n1), tok_idf(S1.nums.values, n1),
            tok_known(S1.core.values), tok_known(S1.addr.values))
    R['cat'] = [c.replace(' ', '') if c else w for c, w in zip(R.core.values, R.web.values)]
    R['chain'] = R.core.map(S1.core.value_counts()).fillna(0).values.astype(np.float32)  # chain / franchise size
    R['fnum'] = np.array([first_num(s) for s in R.nums.values], dtype=np.int64)
    # v7: unmatched Source 2/3 records are records of businesses that are NOT in Source 1, and those businesses
    # have several records of their own. So a name (or name + number, or number + street) that repeats across
    # Source 2/3 but differs from the Source 1 candidate points to a separate business, not to a typo.
    score = [' '.join(sorted(c.split())) for c in R.core.values]
    fn = [s.split()[0] if s else '' for s in R.nums.values]
    sig = [a + '|' + b if a and b else '' for a, b in zip(score, fn)]

    def fnum_street(addr):
        toks = addr.split()
        for i, t in enumerate(toks):
            if t.isdigit():
                for u in toks[i + 1:]:
                    if not u.isdigit(): return t + '|' + u
                return t
        return ''
    asig = [fnum_street(a) for a in R.addr.values]
    isq = (R.src.values != 1)
    for col, keys in (('qcnt_name', score), ('qcnt_sig', sig), ('qcnt_addr', asig)):
        s = pd.Series(keys)
        vc = s[isq & (s != '').values].value_counts()
        R[col] = s.map(vc).fillna(0).values.astype(np.float32)
    return idfs


def _feat_chunk(sub, a, b, idf_core, idf_addr, idf_num, kn_core, kn_addr):
    V = {'name': HV_C3.transform(sub.name.values), 'core': HV_C3.transform(sub.core.values),
         'cat': HV_C3n.transform(sub.cat.values), 'addr': HV_C3.transform(sub.addr.values),
         'tcore': HV_TOK.transform(sub.core.values), 'taddr': HV_TOK.transform(sub.addr.values),
         'tnum': HV_TOK.transform(sub.nums.values)}
    Va = {k: v[a] for k, v in V.items()}; Vb = {k: v[b] for k, v in V.items()}
    F = {}
    for k in ['name', 'core', 'cat', 'addr']:
        F['cos_' + k] = rowdot(Va[k], Vb[k])
    F.update(set_feats(Va['tcore'], Vb['tcore'], idf_core, 'nt_'))
    F.update(set_feats(Va['taddr'], Vb['taddr'], idf_addr, 'at_'))
    F.update(set_feats(Va['tnum'], Vb['tnum'], idf_num, 'num_'))
    F.update(only_feats(Va['tcore'], Vb['tcore'], idf_core, kn_core, 'nx_'))
    F.update(only_feats(Va['taddr'], Vb['taddr'], idf_addr, kn_addr, 'ax_'))
    ca, cb = sub.core.values[a], sub.core.values[b]
    F['core_eq'] = (ca == cb).astype(np.int8)
    F['core_sorted_eq'] = np.array([sorted(x.split()) == sorted(y.split()) for x, y in zip(ca, cb)], np.int8)
    ta, tb = sub.cat.values[a], sub.cat.values[b]
    webb = sub.web.values[b]
    F['web_lcp'] = np.array([lcp(w, t) / max(len(w), 1) if w else -1 for w, t in zip(webb, ta)], np.float32)
    F['cat_lcp'] = np.array([lcp(x, y) / max(min(len(x), len(y)), 1) for x, y in zip(ta, tb)], np.float32)
    da, db = sub.dcat.values[a], sub.dcat.values[b]
    F['dcat_eq'] = np.where(db == '', -1, (da == db)).astype(np.int8)
    F['dcat_in'] = np.array([-1 if not y else int(y in x or x in y) for x, y in zip(da, db)], np.int8)
    fa, fb = sub.fnum.values[a], sub.fnum.values[b]
    F['hnum_diff'] = np.where((fa < 0) | (fb < 0), -1, np.log1p(np.abs(fa - fb))).astype(np.float32)
    sa, sb = sub.state.values[a], sub.state.values[b]
    F['state_eq'] = np.where((sa == '') | (sb == ''), -1, (sa == sb)).astype(np.int8)
    F['b_addr_empty'] = (sub.addr.values[b] == '').astype(np.int8)
    F['b_native'] = sub.native.values[b]; F['b_dba'] = sub.dba.values[b]
    F['b_web'] = (webb != '').astype(np.int8); F['b_src'] = sub.src.values[b]
    F['a_chain'] = sub.chain.values[a]; F['b_chain'] = sub.chain.values[b]
    F['len_a'] = np.array([len(x) for x in ca], np.int16); F['len_b'] = np.array([len(x) for x in cb], np.int16)
    for col in ('qcnt_name', 'qcnt_sig', 'qcnt_addr'):
        F['a_' + col] = sub[col].values[a]; F['b_' + col] = sub[col].values[b]
    return pd.DataFrame(F).values.astype(np.float16)


def make_features(R, C, idfs, path, chunk=100_000):
    """Writes float16 [CTX_COLS + pairwise features] for every row of C to a memory-mapped .npy file."""
    q, s1 = C.q.values, C.s1.values
    RF = R[FEAT_COLS]
    n, nctx = len(C), len(CTX_COLS)
    X = None
    starts = list(range(0, n, chunk))
    step = max(1, N_JOBS * 2)
    with Parallel(n_jobs=N_JOBS) as par:
        for bi in range(0, len(starts), step):
            bs = starts[bi:bi + step]
            tasks = []
            for s in bs:
                a, b = s1[s:s + chunk], q[s:s + chunk]
                u = np.unique(np.concatenate([a, b]))
                tasks.append(delayed(_feat_chunk)(RF.iloc[u], np.searchsorted(u, a), np.searchsorted(u, b), *idfs))
            parts = par(tasks)
            if X is None:
                X = np.lib.format.open_memmap(path, mode='w+', dtype=np.float16, shape=(n, nctx + parts[0].shape[1]))
            for s, part in zip(bs, parts):
                X[s:s + len(part), nctx:] = part
            del parts, tasks
            if (bi // step) % 20 == 0:
                log(f'    features {min(n, bs[-1] + chunk):,}/{n:,}')
    for s in range(0, n, 2_000_000):
        X[s:s + 2_000_000, :nctx] = C[CTX_COLS].iloc[s:s + 2_000_000].values.astype(np.float16)
    X.flush()
    return X

## 5. Stage-2 "sibling" features, assignment and metric

In [ ]:
def group_feats(q, s1, p, src):
    f32 = np.float32
    out = np.empty((len(q), 13), np.float16)
    D = pd.DataFrame({'q': q, 's1': s1, 'p': p.astype(f32), 'src': src})
    out[:, 0] = D.p.values
    qr = D.groupby('q').p.rank(ascending=False, method='first').values.astype(f32); out[:, 1] = qr
    qmax = D.groupby('q').p.transform('max').values.astype(f32); out[:, 2] = qmax
    q2 = D.p.where(qr == 2, f32(0)).groupby(D.q).transform('max').values.astype(f32)
    out[:, 3] = np.where(qr == 1, q2, qmax); del q2, qmax
    top = qr == 1; out[:, 4] = top; del qr
    pt = (D.p.values * top).astype(f32); out[:, 5] = pt; del top
    D['pt'] = pt; D['pt5'] = (pt > 0.5).astype(np.int8); D.drop(columns=['q', 'p'], inplace=True)
    gs = D.groupby('s1')
    ssum = gs.pt.transform('sum').values.astype(f32); out[:, 6] = ssum
    out[:, 7] = gs.pt5.transform('sum').values
    out[:, 8] = gs.pt.transform('max').values
    gss = D.groupby(['s1', 'src'])
    out[:, 9] = gss.pt5.transform('sum').values
    out[:, 10] = gss.pt.rank(ascending=False, method='first').values
    out[:, 11] = D.pt.where(D.pt > 0.5, f32(9)).groupby([D.s1, D.src]).transform('min').replace(9.0, 0.0).values
    out[:, 12] = ssum - pt
    return out


CAPS = None   # max matches per (S1, source), learned from training ground truth


def assign(q, s1, p, thr, y=None, src=None):
    """Each S2/S3 record goes to its highest-probability S1 candidate if p >= thr.
    v4: an S1 entity never receives more records from one source than ever seen in training."""
    D = pd.DataFrame({'q': q, 's1': s1, 'p': p, 't': thr})
    if y is not None: D['y'] = y
    D = D.sort_values(['q', 'p'], ascending=[True, False]).drop_duplicates('q')
    D = D[D.p >= D.t]
    if src is not None and CAPS:
        D['src'] = src[D.q.values]
        D['r'] = D.groupby(['s1', 'src']).p.rank(ascending=False, method='first')
        D = D[D.r <= D.src.map(CAPS).fillna(99)]
    return D


def macro_f05(A, ntrue, eval_mask):
    n1 = len(ntrue)
    TP = np.bincount(A.s1.values, weights=A.y.values, minlength=n1); NP = np.bincount(A.s1.values, minlength=n1)
    P = np.where(NP > 0, TP / np.maximum(NP, 1), 0); Rc = np.where(ntrue > 0, TP / np.maximum(ntrue, 1), 0)
    F = np.where((ntrue == 0) & (NP == 0), 1.0, np.where(TP > 0, 1.25 * P * Rc / np.maximum(.25 * P + Rc, 1e-9), 0.0))
    return F[eval_mask].mean()


def predict_chunked(models, X, extra=None, chunk=1_000_000):
    out = np.zeros(len(X), np.float32)
    for s in range(0, len(X), chunk):
        Xc = X[s:s + chunk].astype(np.float32)
        if extra is not None: Xc = np.hstack([Xc, extra[s:s + chunk]])
        out[s:s + chunk] = np.mean([m.predict_proba(Xc)[:, 1] for m in models], axis=0)
    return out

## 6. Training data: normalize, block, label

In [ ]:
tr = {k: read_tsv(f'{DATA_DIR}/train/train_source{k}.tsv') for k in (1, 2, 3)}
gt = read_tsv(f'{DATA_DIR}/train/train_ground_truth.tsv')
log('train sizes', {k: len(v) for k, v in tr.items()}, 'gt', len(gt))
TDICT = learn_tdict(tr[1], pd.concat([tr[2], tr[3]]), gt)
pickle.dump(TDICT, open(f'{WORK_DIR}/tdict.pkl', 'wb'))
log('translit dictionary size', len(TDICT))

R, keys, n1 = prep_all(tr, TDICT); del tr; gc.collect()
C = run_blocking(R, keys, n1); del keys; gc.collect()
C = add_ctx(C)

# labels
idx = pd.Index(R.id.values)
gtx = gt.assign(m=gt.matched_entity_ids.str.split(',')).explode('m')
gtx = gtx[gtx.m.notna() & (gtx.m != '')]
truth = np.full(len(R), -1, np.int64)
truth[idx.get_indexer(gtx.m.values)] = idx.get_indexer(gtx.source1_entity_id.values)
ntrue = np.zeros(n1); np.add.at(ntrue, idx.get_indexer(gtx.source1_entity_id.values), 1)
C['y'] = (truth[C.q.values] == C.s1.values).astype(np.int8)
CAPS = {2: int(gt.matched_entity_ids.str.count('S2-').max()), 3: int(gt.matched_entity_ids.str.count('S3-').max())}
log('per-source caps from training GT:', CAPS)
log(f'candidates after pruning: {len(C):,} | blocking recall of true pairs: {C.y.sum() / len(gtx):.4f}')

## 7. Two-fold out-of-fold training (stage 1 and stage 2) + threshold selection
S1 entities are split by a hash into folds 0/1. A subset F (TRAIN_S1_FRAC) is used; every query touching F keeps
all its candidates, so the per-query argmax and the per-entity metric are computed exactly as on the test set.

In [ ]:
h = np.array([zlib.crc32(s.encode()) for s in R.id.values[:n1]], dtype=np.uint64)
fold = (h % 2).astype(np.int8)
inF = ((h >> np.uint64(8)) % np.uint64(10000)).astype(np.float64) / 10000 < TRAIN_S1_FRAC
qsel = np.unique(C.q.values[inF[C.s1.values]])
P = C[np.isin(C.q.values, qsel)].reset_index(drop=True)
del C; gc.collect()
log(f'model pairs: {len(P):,} for {inF.sum():,} S1 entities')

idfs = add_stats(R, n1)
X = make_features(R, P, idfs, os.path.join(TMP_DIR, 'X_train.npy'))
log('features', X.shape)
y = P.y.values; pf = fold[P.s1.values]; pin = inF[P.s1.values]
rng = np.random.default_rng(0)

oof1 = np.zeros(len(P), np.float32); models1 = []
for f in (0, 1):
    rows = np.where(pf == f)[0]
    if len(rows) > MAX_ROWS_PER_FOLD: rows = rng.choice(rows, MAX_ROWS_PER_FOLD, replace=False)
    m = HistGradientBoostingClassifier(**GBM_PARAMS).fit(X[rows].astype(np.float32), y[rows])
    oof1[pf != f] = predict_chunked([m], X[pf != f]); models1.append(m)
    log(f'stage-1 fold {f} trained on {len(rows):,} rows')

G = group_feats(P.q.values, P.s1.values, oof1, R.src.values[P.q.values])
oof2 = np.zeros(len(P), np.float32); models2 = []
for f in (0, 1):
    rows = np.where((pf == f) & pin)[0]
    if len(rows) > MAX_ROWS_PER_FOLD: rows = rng.choice(rows, MAX_ROWS_PER_FOLD, replace=False)
    m = HistGradientBoostingClassifier(**GBM_PARAMS).fit(np.hstack([X[rows].astype(np.float32), G[rows]]), y[rows])
    oof2[pf != f] = predict_chunked([m], X[pf != f], G[pf != f]); models2.append(m)
    log(f'stage-2 fold {f} trained on {len(rows):,} rows')

best = (0, 0.5)
for thr in np.arange(0.30, 0.951, 0.05):
    s1f = macro_f05(assign(P.q.values, P.s1.values, oof1, thr, y), ntrue, inF)
    s2f = macro_f05(assign(P.q.values, P.s1.values, oof2, thr, y, R.src.values), ntrue, inF)
    print(f'thr={thr:.2f}  stage1 F0.5={s1f:.5f}  stage2+caps F0.5={s2f:.5f}')
    if s2f > best[0]: best = (s2f, float(thr))
THR = best[1]
log(f'VALIDATION macro F0.5 (stage 2) = {best[0]:.5f} at threshold {THR:.2f}')
pickle.dump({'models1': models1, 'models2': models2, 'thr': THR, 'caps': CAPS}, open(f'{WORK_DIR}/models.pkl', 'wb'))
del R, P, X, G, oof1, oof2, y, truth; gc.collect()
os.remove(os.path.join(TMP_DIR, 'X_train.npy'))

## 8. Test inference

In [ ]:
te = {k: read_tsv(f'{DATA_DIR}/test/test_source{k}.tsv') for k in (1, 2, 3)}
log('test sizes', {k: len(v) for k, v in te.items()}, 'countries', te[1].country.value_counts().to_dict())
s1_order = te[1].entity_id.values
R, keys, n1 = prep_all(te, TDICT); del te; gc.collect()
C = run_blocking(R, keys, n1); del keys; gc.collect()
C = add_ctx(C)
idfs = add_stats(R, n1)
X = make_features(R, C, idfs, os.path.join(TMP_DIR, 'X_test.npy'))
log('test features', X.shape)
R = R[['id', 'src', 'country']]; gc.collect()   # free the text columns
p1 = predict_chunked(models1[:TEST_MODELS], X); log('stage-1 predicted')
G = group_feats(C.q.values, C.s1.values, p1, R.src.values[C.q.values])
p2 = predict_chunked(models2[:TEST_MODELS], X, G); log('stage-2 predicted')
del G; gc.collect()
s1_country = R.country.values[C.s1.values]
thr_arr = np.where(s1_country == 'france', THR_FRANCE if THR_FRANCE is not None else THR, THR)
A = assign(C.q.values, C.s1.values, p2, thr_arr, src=R.src.values)
# save predictions so thresholds can be changed later without re-running the pipeline
pickle.dump({'q': C.q.values, 's1': C.s1.values, 'p2': p2, 'ids': R.id.values, 'n1': n1,
             'country': R.country.values[:n1], 's1_order': s1_order, 'thr': THR, 'caps': CAPS,
             'src_q': R.src.values},
            open(f'{WORK_DIR}/test_preds.pkl', 'wb'))
log(f'assigned {len(A):,} S2/S3 records to {A.s1.nunique():,} S1 entities')

## 9. Write outputs and validate

In [ ]:
ids = R.id.values
assert (ids[:n1] == s1_order).all()   # S1 rows of R are in test_source1 file order


def write_lists(path, col, s1_idx, q_idx):
    order = np.argsort(s1_idx, kind='stable'); ss = s1_idx[order]; qq = q_idx[order]
    bounds = np.searchsorted(ss, np.arange(n1 + 1))
    with open(path, 'w') as f:
        f.write(f'source1_entity_id\t{col}\n')
        for i in range(n1):
            a, b = bounds[i], bounds[i + 1]
            f.write(ids[i] + '\t' + (','.join(ids[qq[a:b]]) if b > a else '') + '\n')


write_lists(f'{OUT_DIR}/matching_results.tsv', 'matched_entity_ids', A.s1.values, A.q.values)
write_lists(f'{OUT_DIR}/candidate_pairs.tsv', 'candidate_entity_ids', C.s1.values, C.q.values)
log('files written')
os.makedirs(f'{OUT_DIR}/variants', exist_ok=True)
for t in VARIANT_THRS:
    Av = assign(C.q.values, C.s1.values, p2, t, src=R.src.values)
    write_lists(f'{OUT_DIR}/variants/matching_results_t{int(round(t * 100))}.tsv', 'matched_entity_ids', Av.s1.values, Av.q.values)
    log(f'  variant thr={t}: {len(Av):,} records assigned')
del Av

# ---- self-check against the submission rules (candidate file is correct by construction: one row per S1,
#      unique (S1, record) pairs from blocking, and every match is one of these pairs) ----
problems = []
out_m = read_tsv(f'{OUT_DIR}/matching_results.tsv')
if list(out_m.columns) != ['source1_entity_id', 'matched_entity_ids']: problems.append('bad header')
if len(out_m) != n1 or out_m.source1_entity_id.duplicated().any(): problems.append('S1 rows missing/duplicated')
valid_q = set(ids[n1:])
lists = [x.split(',') for x in out_m.matched_entity_ids.values if x]
if any(len(l) != len(set(l)) for l in lists): problems.append('duplicate ids in a list')
if any(i not in valid_q for l in lists for i in l): problems.append('unknown or non S2/S3 id')
if A.q.duplicated().any(): problems.append('a record assigned to two S1 entities')
log('CHECK:', 'PASS' if not problems else problems)
nm = np.bincount(A.s1.values, minlength=n1)
print(pd.DataFrame({'country': R.country.values[:n1], 'has_match': nm > 0, 'n_match': nm}).groupby('country').mean())
print(out_m.head())